# tw-med-llm-qlora — optional GGUF Q4_K_M export

This Linux Colab notebook is **not required** for adapter delivery. It is
approved in repository config but remains disabled in the first code cell for
every new run. After copying the exact approval code, use an A100 40GB runtime
and execute from the top. It never pushes a model; each run writes to a unique
Drive directory for later Windows Ollama validation.


In [ ]:
# OPTIONAL EXPORT HARD GATE — this is the only cell that may be edited.
CONFIG_EXPORT_ENABLED = True
ENABLE_GGUF_EXPORT = False
GGUF_EXPORT_APPROVAL = ""
REQUIRED_GGUF_EXPORT_APPROVAL = 'GGUF_Q4_K_M_A100_APPROVED_20260723'
APPROVED_COMPUTE_UNIT_LIMIT = 6.36

if not CONFIG_EXPORT_ENABLED:
    raise RuntimeError(
        "Repo config keeps optional GGUF export disabled. Obtain explicit approval, "
        "update configs/project.toml, and rebuild this notebook first."
    )
if not ENABLE_GGUF_EXPORT or GGUF_EXPORT_APPROVAL != REQUIRED_GGUF_EXPORT_APPROVAL:
    raise RuntimeError(
        "GGUF export is locked. Set ENABLE_GGUF_EXPORT=True and copy the exact "
        "approval code in this cell only after the repo gate is enabled."
    )
print(
    "GGUF Q4_K_M export gate passed. "
    f"Approved compute-unit limit: {APPROVED_COMPUTE_UNIT_LIMIT:.2f} CU"
)


## 1. Install the pinned Phase 3-compatible export stack


In [ ]:
%pip install --quiet \
    "unsloth[colab-new]==2026.7.4" \
    "unsloth-zoo==2026.7.4" \
    "transformers==4.56.2" \
    "peft==0.19.1" \
    "bitsandbytes==0.49.2" \
    "accelerate==1.14.0" \
    "huggingface-hub==0.35.3" \
    "sentencepiece==0.2.2" \
    "protobuf==6.33.5"


## 2. Verify A100, Drive, token, and disk


In [ ]:
# ruff: noqa: E501
import hashlib
import importlib.metadata
import json
import shutil
import time
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import torch
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

PROJECT_CONFIG = json.loads('{"data": {"medqa": {"config": "med_qa_tw_source", "dataset_id": "bigbio/med_qa", "expected_test_rows": 1413, "expected_train_rows": 11248, "expected_validation_rows": 1409, "revision": "e04abdc0672c54547fa1dbe36cfefc000e4f2657", "sample_size": 5, "source_sha256": {"test": "799f6c7881d50b1101f168f31129aea699fec67401e2b8dc9464f1f84dc40c04", "train": "b2cb4b82a7c7ffc6f6f99465197f4dfc94b04464cf2ab517a67418641c617646", "validation": "06c2334173f17ed0665088e920bbf0fb54bb64ef7432885747668235ed54701a"}}, "tmmluplus": {"dataset_id": "ikala/tmmluplus", "revision": "81d53e38340c9ade988f7fed8996da6554b504f3"}}, "evaluation": {"bootstrap_iterations": 10000, "calibration_examples": 20, "control_subjects": ["chinese_language_and_literature", "geography_of_taiwan", "logic_reasoning", "computer_science", "general_principles_of_law"], "forgetting_margin_percentage_points": 2.0, "full_approval": {"approval_phrase": "確認解鎖 Phase 4 正式評估", "approved": true, "approved_at": "2026-07-22", "approved_requests": 28758, "calibration_manifest_sha256": "d9c0f23e72808f0a8fc4edc1a7889719637f9390346725d5f72f10c4d3e2cdf2", "calibration_run_id": "20260722T061028Z", "calibration_validation_sha256": "6c39aff8cd4e82b80d24a4ce7f7959e7b7fd5ba9546f2432ef3b7c96212b2d85", "compute_units_per_hour": 5.3, "parallel_workers": 4, "projected_compute_units": 15.791642498050743, "projected_compute_units_with_20pct_buffer": 18.94997099766089, "projected_hours": 2.979555188311461, "required_approval_code": "PHASE4_FULL_28758_APPROVED_20260722", "shard_size": 250}, "full_shuffle_seed": 3407, "generation": {"max_tokens": 256, "minimum_calibration_parse_rate": 0.8, "temperature": 0.0, "token_limit_hits_count_as_incorrect": true, "token_limit_hits_fail_calibration": false, "top_p": 1.0}, "medical_subjects": ["basic_medical_science", "clinical_psychology", "dentistry", "occupational_therapy_for_psychological_disorders", "optometry", "pharmacology", "pharmacy", "traditional_chinese_medicine_clinical_medicine"], "phase3_adapter": {"archive_bytes": 113079186, "archive_sha256": "2c537dfd3049319286c678a3ca3aa72e3f20baa7e0f44bde93ff7ee4dc47e43e", "base_model_id": "taide/Gemma-3-TAIDE-12b-Chat-2602", "drive_archive": "/content/drive/MyDrive/tw-med-llm-qlora/phase3/141226999eacb67ffd9a/full/runs/20260722T014216Z-phase3-full.zip", "selected_checkpoint": 700}, "stability_examples_per_subject": 100, "stability_seeds": [3407, 3408, 3409], "twinkle_eval": {"extractor": "box", "repository": "https://github.com/ai-twinkle/Eval", "revision": "470bbec130fa95c9e2e06c6a4b06db4a51390a06", "scorer": "exact_match", "version": "2.8.0"}, "vllm": {"expected_torch_cuda": "12.9", "gpu_memory_utilization": 0.85, "language_model_only": true, "max_lora_rank": 16, "max_model_length": 2048, "quantization": "bitsandbytes", "torch_backend": "cu129", "uv_version": "0.11.31", "version": "0.25.1", "wheel_sha256": "9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431", "wheel_url": "https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl", "wheel_variant": "cu129"}, "workload": {"expected_medqa_full_requests": 4239, "expected_tmmlu_full_requests": 16719, "expected_tmmlu_stability_requests": 7800, "expected_total_requests": 28758, "full_model_count": 3, "medqa_test_rows": 1413, "stability_model_count": 2, "tmmlu_test_rows": 5573}}, "export": {"gguf": {"approval_phrase": "確認解鎖 GGUF Q4_K_M A100 匯出，上限 6.36 CU；產物只存 Google Drive，不上傳外部，完成後執行 Windows Ollama 驗收", "approved_at": "2026-07-23", "approved_compute_units_with_20pct_buffer": 6.36, "compute_units_per_hour": 5.3, "drive_root": "/content/drive/MyDrive/tw-med-llm-qlora/phase5/gguf-q4-k-m/runs", "enabled": true, "expected_gguf_gib_lower": 7.0, "expected_gguf_gib_upper": 9.0, "external_upload_allowed": false, "maximum_memory_usage": 0.5, "minimum_drive_disk_gib": 20.0, "minimum_local_disk_gib": 100.0, "minimum_vram_gib": 38.0, "ollama_model_name": "tw-med-taide-12b-q4-k-m", "projected_compute_units_lower": 2.65, "projected_compute_units_upper": 5.3, "projected_hours_lower": 0.5, "projected_hours_upper": 1.0, "quantization_method": "q4_k_m", "required_approval_code": "GGUF_Q4_K_M_A100_APPROVED_20260723", "required_gpu_name_contains": "A100", "run_environment": "colab_linux"}}, "hardware_profiles": [{"allow_tf32": true, "batch_size": 8, "gradient_accumulation_steps": 2, "max_sequence_length": 2048, "min_compute_capability": [8, 0], "min_vram_gib": 70.0, "model_profile": "primary", "name": "primary_80g", "requires_bf16": true}, {"allow_tf32": true, "batch_size": 4, "gradient_accumulation_steps": 4, "max_sequence_length": 2048, "min_compute_capability": [8, 0], "min_vram_gib": 38.0, "model_profile": "primary", "name": "primary_40g", "requires_bf16": true}, {"allow_tf32": true, "batch_size": 1, "gradient_accumulation_steps": 16, "max_sequence_length": 2048, "min_compute_capability": [8, 0], "min_vram_gib": 22.0, "model_profile": "primary", "name": "primary_24g", "requires_bf16": true}, {"allow_tf32": false, "batch_size": 2, "gradient_accumulation_steps": 8, "max_sequence_length": 2048, "min_compute_capability": [7, 5], "min_vram_gib": 14.0, "model_profile": "fallback", "name": "fallback_16g", "requires_bf16": false}], "inference": {"phase3_adapter": {"archive_sha256": "2c537dfd3049319286c678a3ca3aa72e3f20baa7e0f44bde93ff7ee4dc47e43e", "base_model_id": "taide/Gemma-3-TAIDE-12b-Chat-2602", "base_model_revision": "4de0b93b99f8b61b59c40d019fd593bdd1c42249", "selected_checkpoint": 700}, "windows_4090": {"attention_implementation": "sdpa", "double_quantization": true, "load_in_4bit": true, "max_new_tokens": 64, "minimum_compute_capability": [8, 9], "minimum_vram_gib": 22.0, "ollama_model_name": "tw-med-taide-12b", "quantization_type": "nf4", "required_gpu_name": "RTX 4090", "requires_bf16": true}}, "models": {"fallback": {"baseline_id": "google/gemma-3-4b-it", "baseline_revision": "093f9f388b31de276ce2de164bdc2081324b9767", "batch_size": 2, "gradient_accumulation_steps": 8, "max_sequence_length": 2048, "model_id": "twinkle-ai/gemma-3-4B-T1-it", "requires_bf16": false, "revision": "63b2bb819a7885b476c2ff98a418de8400661f9d"}, "primary": {"baseline_id": "google/gemma-3-12b-it", "baseline_revision": "96b6f1eccf38110c56df3a15bffe176da04bfd80", "batch_size": 1, "gradient_accumulation_steps": 16, "max_sequence_length": 2048, "model_id": "taide/Gemma-3-TAIDE-12b-Chat-2602", "requires_bf16": true, "revision": "4de0b93b99f8b61b59c40d019fd593bdd1c42249"}}, "project": {"effective_batch_size": 16, "seed": 3407}, "publication": {"adapter_repo_id": "steven0226/tw-med-llm-qlora-adapter", "enabled": true, "github_repository_url": "https://github.com/kuotunyu/tw-med-llm-qlora", "requires_explicit_repo_id": true, "requires_explicit_visibility_confirmation": true, "visibility": "private"}, "training": {"eval_steps": 100, "learning_rate": 5e-05, "load_in_4bit": true, "lora_alpha": 16, "lora_dropout": 0.0, "lora_rank": 16, "lr_scheduler_type": "cosine", "num_train_epochs": 1, "optimizer": "adamw_8bit", "save_steps": 100, "save_total_limit": 2, "smoke_examples": 100, "smoke_steps": 10, "warmup_ratio": 0.03}}')
export_config = PROJECT_CONFIG["export"]["gguf"]
if not export_config["enabled"]:
    raise RuntimeError("Embedded repository export gate is disabled")
if export_config["required_approval_code"] != REQUIRED_GGUF_EXPORT_APPROVAL:
    raise RuntimeError("Notebook approval code does not match embedded repository config")
if export_config["quantization_method"] != "q4_k_m":
    raise RuntimeError("Only the approved q4_k_m quantization method is allowed")
if export_config["run_environment"] != "colab_linux":
    raise RuntimeError("GGUF export is restricted to the approved Colab Linux environment")
if export_config["external_upload_allowed"]:
    raise RuntimeError("External upload must remain disabled for this export")
if (
    float(export_config["approved_compute_units_with_20pct_buffer"])
    != APPROVED_COMPUTE_UNIT_LIMIT
):
    raise RuntimeError("Approved compute-unit limit does not match repository config")

workflow_started_at_utc = datetime.now(UTC)
workflow_started_monotonic = time.perf_counter()
run_id = workflow_started_at_utc.strftime("%Y%m%dT%H%M%SZ")
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Colab Secret HF_TOKEN is required")
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required")
properties = torch.cuda.get_device_properties(0)
total_vram_gib = properties.total_memory / 1024**3
gpu = {
    "name": properties.name,
    "total_vram_gib": total_vram_gib,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
}
expected_gpu_name = str(export_config["required_gpu_name_contains"])
if expected_gpu_name.casefold() not in properties.name.casefold():
    raise RuntimeError(
        f"Only the approved {expected_gpu_name} profile may run this export: {gpu}"
    )
minimum_vram_gib = float(export_config["minimum_vram_gib"])
if total_vram_gib < minimum_vram_gib or not gpu["bf16_supported"]:
    raise RuntimeError(
        f"A BF16 GPU with at least {minimum_vram_gib:.1f} GiB is required: {gpu}"
    )
free_disk_gib = shutil.disk_usage("/content").free / 1024**3
minimum_local_disk_gib = float(export_config["minimum_local_disk_gib"])
if free_disk_gib < minimum_local_disk_gib:
    raise RuntimeError(
        f"At least {minimum_local_disk_gib:.1f} GiB free local disk is required: "
        f"{free_disk_gib:.1f}"
    )
drive.mount("/content/drive")
phase3 = PROJECT_CONFIG["evaluation"]["phase3_adapter"]
archive_path = Path(phase3["drive_archive"])
drive_root = Path(export_config["drive_root"])
drive_root.mkdir(parents=True, exist_ok=True)
free_drive_gib = shutil.disk_usage(drive_root).free / 1024**3
minimum_drive_disk_gib = float(export_config["minimum_drive_disk_gib"])
if free_drive_gib < minimum_drive_disk_gib:
    raise RuntimeError(
        f"At least {minimum_drive_disk_gib:.1f} GiB free Drive space is required: "
        f"{free_drive_gib:.1f}"
    )
drive_output = drive_root / run_id
if drive_output.exists():
    raise FileExistsError(f"Refusing to overwrite an existing export run: {drive_output}")
local_root = Path("/content/tw-med-gguf-export") / run_id
adapter_dir = local_root / "adapter"
export_dir = local_root / "q4_k_m"
for directory in (local_root, adapter_dir, export_dir):
    directory.mkdir(parents=True, exist_ok=False)
print(
    json.dumps(
        {
            "run_id": run_id,
            "gpu": gpu,
            "free_local_disk_gib": free_disk_gib,
            "free_drive_disk_gib": free_drive_gib,
            "approved_compute_unit_limit": APPROVED_COMPUTE_UNIT_LIMIT,
        },
        indent=2,
    )
)


## 3. Verify and extract only the selected step-700 adapter


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not archive_path.is_file():
    raise FileNotFoundError(f"Phase 3 full archive not found: {archive_path}")
if archive_path.stat().st_size != int(phase3["archive_bytes"]):
    raise RuntimeError(
        f"Phase 3 archive size mismatch: {archive_path.stat().st_size} != "
        f"{phase3['archive_bytes']}"
    )
archive_sha256 = sha256_file(archive_path)
if archive_sha256 != phase3["archive_sha256"]:
    raise RuntimeError(
        f"Phase 3 archive SHA-256 mismatch: {archive_sha256} != "
        f"{phase3['archive_sha256']}"
    )
with ZipFile(archive_path) as archive:
    selected = []
    for member in archive.infolist():
        normalized = PurePosixPath(member.filename.replace("\\", "/"))
        if normalized.is_absolute() or ".." in normalized.parts:
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        if normalized.parts and normalized.parts[0] == "adapter" and not member.is_dir():
            selected.append((member, PurePosixPath(*normalized.parts[1:])))
    names = {relative.as_posix() for _, relative in selected}
    required = {"adapter_config.json", "adapter_model.safetensors"}
    if not required.issubset(names):
        raise RuntimeError(f"Phase 3 archive is missing adapter files: {required - names}")
    for member, relative in selected:
        target = adapter_dir.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open("wb") as output:
            shutil.copyfileobj(source, output)
adapter_config = json.loads(
    (adapter_dir / "adapter_config.json").read_text(encoding="utf-8")
)
model_config = PROJECT_CONFIG["models"]["primary"]
if adapter_config.get("base_model_name_or_path") != model_config["model_id"]:
    raise RuntimeError("Adapter/base mismatch; export stopped")
print({"adapter_files": len(selected), "archive_sha256_verified": True})


## 4. Download, verify, and load the pinned base and frozen adapter

This cell first completes the exact Hugging Face snapshot and verifies every
safetensors shard, tokenizer, and required processor file. It then gives
Unsloth the verified local snapshot only, preventing an interrupted cache
download from being mistaken for a loadable model.


In [ ]:
# ruff: noqa: I001  # Unsloth must patch Transformers before PEFT imports it.
from unsloth import FastModel
from peft import PeftModel

model_id = str(model_config["model_id"])
model_revision = str(model_config["revision"])
model_info = HfApi(token=HF_TOKEN).model_info(
    repo_id=model_id,
    revision=model_revision,
    files_metadata=True,
)
if model_info.sha != model_revision:
    raise RuntimeError(
        "Resolved model revision mismatch: "
        f"expected={model_revision}, actual={model_info.sha}"
    )

model_siblings = model_info.siblings or []
remote_weight_files = sorted(
    sibling.rfilename
    for sibling in model_siblings
    if sibling.rfilename.endswith(".safetensors")
)
remote_weight_bytes = sum(
    int(sibling.size or 0)
    for sibling in model_siblings
    if sibling.rfilename.endswith(".safetensors")
)
if not remote_weight_files or remote_weight_bytes <= 0:
    raise RuntimeError(
        "Hugging Face model metadata has no sized safetensors weights."
    )

free_disk_bytes_before_download = shutil.disk_usage("/content").free
required_disk_bytes = remote_weight_bytes + 8 * 1024**3
if free_disk_bytes_before_download < required_disk_bytes:
    raise RuntimeError(
        "Colab local disk is too small for the pinned model snapshot: "
        f"free={free_disk_bytes_before_download / 1024**3:.2f} GiB, "
        f"required={required_disk_bytes / 1024**3:.2f} GiB. "
        "Factory-reset the runtime or choose a runtime with more local disk."
    )

snapshot_path = Path(
    snapshot_download(
        repo_id=model_id,
        revision=model_revision,
        token=HF_TOKEN,
        allow_patterns=[
            "*.json",
            "*.jinja",
            "*.model",
            "*.safetensors",
            "*.txt",
        ],
        max_workers=8,
    )
)
if snapshot_path.name != model_revision:
    raise RuntimeError(
        "Downloaded snapshot revision mismatch: "
        f"expected={model_revision}, actual={snapshot_path.name}"
    )

missing_remote_weights = [
    filename
    for filename in remote_weight_files
    if not (snapshot_path / filename).is_file()
    or (snapshot_path / filename).stat().st_size <= 0
]
if missing_remote_weights:
    raise RuntimeError(
        "Incomplete model snapshot; missing or empty weights: "
        f"{missing_remote_weights}"
    )

index_path = snapshot_path / "model.safetensors.index.json"
single_weight_path = snapshot_path / "model.safetensors"
if index_path.is_file():
    weight_index = json.loads(index_path.read_text(encoding="utf-8"))
    indexed_shards = sorted(set(weight_index.get("weight_map", {}).values()))
    if not indexed_shards:
        raise RuntimeError("model.safetensors.index.json has an empty weight_map")
    unsafe_shards = [
        filename
        for filename in indexed_shards
        if Path(filename).name != filename
        or not filename.endswith(".safetensors")
    ]
    if unsafe_shards:
        raise RuntimeError(f"Unsafe shard names in model index: {unsafe_shards}")
    missing_indexed_shards = [
        filename
        for filename in indexed_shards
        if not (snapshot_path / filename).is_file()
        or (snapshot_path / filename).stat().st_size <= 0
    ]
    if missing_indexed_shards:
        raise RuntimeError(
            "Model index references missing or empty shards: "
            f"{missing_indexed_shards}"
        )
elif single_weight_path.is_file() and single_weight_path.stat().st_size > 0:
    indexed_shards = [single_weight_path.name]
else:
    raise RuntimeError(
        "Pinned snapshot has neither model.safetensors nor "
        "model.safetensors.index.json"
    )

snapshot_config_path = snapshot_path / "config.json"
tokenizer_config_path = snapshot_path / "tokenizer_config.json"
if not snapshot_config_path.is_file() or not tokenizer_config_path.is_file():
    raise RuntimeError(
        "Pinned snapshot is missing config.json or tokenizer_config.json"
    )
if not any(
    (snapshot_path / filename).is_file()
    and (snapshot_path / filename).stat().st_size > 0
    for filename in ("tokenizer.json", "tokenizer.model")
):
    raise RuntimeError(
        "Pinned snapshot has neither a usable tokenizer.json nor tokenizer.model"
    )
snapshot_config = json.loads(snapshot_config_path.read_text(encoding="utf-8"))
snapshot_architectures = snapshot_config.get("architectures") or []
snapshot_is_vlm = bool(snapshot_config.get("vision_config")) or any(
    str(architecture).endswith("ForConditionalGeneration")
    for architecture in snapshot_architectures
)
if snapshot_is_vlm:
    missing_processor_files = [
        filename
        for filename in ("processor_config.json", "preprocessor_config.json")
        if not (snapshot_path / filename).is_file()
        or (snapshot_path / filename).stat().st_size <= 0
    ]
    if missing_processor_files:
        raise RuntimeError(
            "VLM snapshot is missing processor files: "
            f"{missing_processor_files}"
        )

MODEL_SNAPSHOT_AUDIT = {
    "repo_id": model_id,
    "revision": model_revision,
    "snapshot_path": str(snapshot_path),
    "remote_weight_files": remote_weight_files,
    "indexed_shards": indexed_shards,
    "weight_bytes": remote_weight_bytes,
    "weight_gib": remote_weight_bytes / 1024**3,
    "free_disk_gib_before_download": free_disk_bytes_before_download / 1024**3,
    "tokenizer_source": str(snapshot_path),
    "vlm_processor_required": snapshot_is_vlm,
    "complete": True,
}
print(json.dumps(MODEL_SNAPSHOT_AUDIT, ensure_ascii=False, indent=2))

base_model, processor = FastModel.from_pretrained(
    model_name=str(snapshot_path),
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    token=HF_TOKEN,
    tokenizer_name=str(snapshot_path),
    local_files_only=True,
    use_safetensors=True,
)
model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
    is_trainable=False,
    low_cpu_mem_usage=True,
    local_files_only=True,
)
model.eval()
trainable, total = model.get_nb_trainable_parameters()
if trainable != 0 or total <= 0:
    raise RuntimeError(f"Frozen adapter audit failed: trainable={trainable}, total={total}")
if not hasattr(model, "save_pretrained_gguf"):
    raise RuntimeError(
        "Current Unsloth no longer exposes save_pretrained_gguf on this PEFT model; "
        "stop and recheck the official API instead of using an unverified fallback."
    )
text_tokenizer = getattr(processor, "tokenizer", processor)
chat_template = (
    getattr(text_tokenizer, "chat_template", None)
    or getattr(processor, "chat_template", None)
)
if not isinstance(chat_template, str) or not chat_template.strip():
    raise RuntimeError("The loaded processor/tokenizer has no chat template")
chat_template_sha256 = hashlib.sha256(chat_template.encode("utf-8")).hexdigest()
print(
    {
        "adapter_reloaded": True,
        "trainable_parameters": trainable,
        "processor_class": processor.__class__.__name__,
        "tokenizer_class": text_tokenizer.__class__.__name__,
        "chat_template_sha256": chat_template_sha256,
    }
)


## 5. Export Q4_K_M and an Ollama Modelfile


In [ ]:
model.save_pretrained_gguf(
    str(export_dir),
    tokenizer=processor,
    quantization_method=export_config["quantization_method"],
    maximum_memory_usage=float(export_config["maximum_memory_usage"]),
)
gguf_files = sorted(export_dir.glob("*.gguf"))
if len(gguf_files) != 1 or gguf_files[0].stat().st_size <= 0:
    raise RuntimeError(f"Expected exactly one non-empty GGUF file: {gguf_files}")
gguf_path = gguf_files[0]
gguf_gib = gguf_path.stat().st_size / 1024**3
expected_gguf_gib = [
    float(export_config["expected_gguf_gib_lower"]),
    float(export_config["expected_gguf_gib_upper"]),
]
if gguf_gib < 4 or gguf_gib > 20:
    raise RuntimeError(f"Implausible 12B Q4_K_M GGUF size: {gguf_gib:.2f} GiB")
expected_size_range_match = expected_gguf_gib[0] <= gguf_gib <= expected_gguf_gib[1]
modelfile = export_dir / "Modelfile"
modelfile.write_text(
    "\n".join(
        [
            f"FROM ./{gguf_path.name}",
            "PARAMETER temperature 0",
            "PARAMETER seed 3407",
            "PARAMETER num_ctx 2048",
            "PARAMETER num_predict 64",
            "",
        ]
    ),
    encoding="utf-8",
    newline="\n",
)
drive_output.mkdir(parents=False, exist_ok=False)
copied = {}
for source in (gguf_path, modelfile):
    partial = drive_output / (source.name + ".partial")
    destination = drive_output / source.name
    if partial.exists() or destination.exists():
        raise FileExistsError(f"Refusing to overwrite Drive output: {destination}")
    shutil.copy2(source, partial)
    source_sha256 = sha256_file(source)
    if partial.stat().st_size != source.stat().st_size:
        raise RuntimeError(f"Drive copy size mismatch: {source.name}")
    if sha256_file(partial) != source_sha256:
        raise RuntimeError(f"Drive copy hash mismatch: {source.name}")
    partial.rename(destination)
    copied[source.name] = {
        "sha256": sha256_file(destination),
        "bytes": destination.stat().st_size,
    }
workflow_elapsed_seconds = time.perf_counter() - workflow_started_monotonic
receipt = {
    "schema_version": 2,
    "phase": 5,
    "optional_export": "gguf_q4_k_m",
    "run_id": run_id,
    "created_at_utc": datetime.now(UTC).isoformat(),
    "workflow_elapsed_seconds": workflow_elapsed_seconds,
    "approval": {
        "approved_at": export_config["approved_at"],
        "required_approval_code": export_config["required_approval_code"],
        "approved_compute_units_with_20pct_buffer": (
            export_config["approved_compute_units_with_20pct_buffer"]
        ),
    },
    "base_model_id": model_config["model_id"],
    "base_model_revision": model_config["revision"],
    "model_snapshot": {
        "resolved_revision": model_info.sha,
        "remote_weight_files": len(remote_weight_files),
        "indexed_shards": len(indexed_shards),
        "weight_bytes": remote_weight_bytes,
        "complete": True,
    },
    "adapter_checkpoint": int(phase3["selected_checkpoint"]),
    "phase3_archive_sha256": archive_sha256,
    "quantization_method": export_config["quantization_method"],
    "maximum_memory_usage": export_config["maximum_memory_usage"],
    "gpu": gpu,
    "resources": {
        "free_local_disk_gib_before_export": free_disk_gib,
        "free_drive_disk_gib_before_export": free_drive_gib,
        "gguf_gib": gguf_gib,
        "expected_gguf_gib": expected_gguf_gib,
        "expected_size_range_match": expected_size_range_match,
    },
    "tokenizer": {
        "processor_class": processor.__class__.__name__,
        "tokenizer_class": text_tokenizer.__class__.__name__,
        "chat_template_sha256": chat_template_sha256,
        "bos_token": text_tokenizer.bos_token,
        "eos_token": text_tokenizer.eos_token,
    },
    "packages": {
        package: importlib.metadata.version(package)
        for package in (
            "unsloth",
            "unsloth-zoo",
            "transformers",
            "peft",
            "bitsandbytes",
            "accelerate",
        )
    },
    "files": copied,
    "chat_template_warning": "Validate the exported GGUF in Ollama before use.",
    "external_upload_performed": False,
    "published": False,
}
receipt_path = drive_output / "gguf-export-receipt.json"
receipt_path.write_text(
    json.dumps(receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
    newline="\n",
)
print(json.dumps(receipt, ensure_ascii=False, indent=2))
print(f"Drive export directory: {drive_output}")
print("Optional export complete. No model was uploaded or published.")
